# ex03 · 多输入多输出通道（对应教材 6.4）

> **做题流程**：从零实现多输入通道 → 多输出通道，再理解 1×1 卷积与参数量。
> **做完再看** `solutions/ex03-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）

In [1]:
import torch
from torch import nn

def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

## 题 1 🔧 从零实现多输入通道（TODO 6.4）

多输入通道 = 每个通道一个核，各自互相关再**相加**。补全 corr2d_multi_in。
（手算对照：X 两通道、K 两通道，结果应为 [[56,72],[104,120]]）

In [2]:
def corr2d_multi_in(X, K):
    # TODO 6.4: sum(corr2d(x, k) for x, k in zip(X, K))
    return sum(corr2d(x, k) for x, k in zip(X, K))

In [3]:
try:
    X = torch.tensor([[[0.,1.,2.],[3.,4.,5.],[6.,7.,8.]],
                      [[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]]])
    K = torch.tensor([[[0.,1.],[2.,3.]], [[1.,2.],[3.,4.]]])
    Y = corr2d_multi_in(X, K)
    assert Y.tolist() == [[56.0, 72.0], [104.0, 120.0]], f'{Y.tolist()}'
    print('✓ corr2d_multi_in 正确:', Y.tolist())
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ corr2d_multi_in 正确: [[56.0, 72.0], [104.0, 120.0]]


## 题 2 🔧 从零实现多输出通道（TODO 6.5）

多输出通道 = 多组核，每组核各算一个输出通道，最后 **stack** 起来。补全 corr2d_multi_in_out。

In [6]:
def corr2d_multi_in_out(X, K):
    # TODO 6.5: 对 K 的每个输出通道（K 的第 0 维）做 multi_in，stack 起来
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

In [7]:
try:
    X = torch.tensor([[[0.,1.,2.],[3.,4.,5.],[6.,7.,8.]],
                      [[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]]])
    K = torch.tensor([[[0.,1.],[2.,3.]], [[1.,2.],[3.,4.]]])
    K3 = torch.stack((K, K + 1, K + 2), 0)   # 3 个输出通道
    Y = corr2d_multi_in_out(X, K3)
    assert list(Y.shape) == [3, 2, 2], f'{Y.shape}'
    print('✓ 多输出通道形状:', list(Y.shape))
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ 多输出通道形状: [3, 2, 2]


## 题 3 🔧 参数量与 1×1 卷积

先回答：`nn.Conv2d(in_ch, out_ch, k)` 的参数量？1×1 卷积在做什么？

逐位置的通道线性变换，用于升降通道数、跨通道融合

In [8]:
conv = nn.Conv2d(3, 8, kernel_size=5)
print('Conv2d(3,8,5) 参数量:', sum(p.numel() for p in conv.parameters()))

conv1x1 = nn.Conv2d(3, 8, kernel_size=1)
X = torch.randn(2, 3, 7, 7)
print('1×1 卷积输出形状:', list(conv1x1(X).shape), '（空间尺寸不变，通道 3→8）')

Conv2d(3,8,5) 参数量: 608
1×1 卷积输出形状: [2, 8, 7, 7] （空间尺寸不变，通道 3→8）


## 小结与面试衔接

- 多输入通道：每通道一核、结果相加；多输出通道：多组核、结果 stack
- 参数量 = in_ch × out_ch × k × k + out_ch
- 1×1 卷积 = 逐位置的通道线性变换，用于升降通道数、跨通道融合（面试高频）